In [40]:
import ast
import pandas as pd

# 1. Prevent Pandas from wrapping columns into multiple lines
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)        # Set a wide display threshold
pd.set_option('display.max_rows', None)     # Show all rows

# Load a 50,000-row sample first (full file has ~1M rows)
#df = pd.read_csv('SGJobData.csv', nrows=50000)
df = pd.read_csv('SGJobData.csv')

print(df.head())
print(df.shape)   # (rows, columns)

# 1. Structural Description: View column names, non-null counts, and data types
print("--- STRUCTURAL INFO ---")
df.info()

# 2. Statistical Description (Numeric columns): View count, mean, min, max, and percentiles
print("\n--- NUMERIC COLUMNS DESCRIBE ---")
print(df.describe())

# 3. Text/Categorical Description (String columns): View count, unique values, top item, and frequency
print("\n--- TEXT/CATEGORICAL COLUMNS DESCRIBE ---")
print(df.describe(include=['object']))

'''
#unique values and their exact counts for each column
for column in df.columns:
    print(f"=== Unique Values for Column: {column} ===")
    print(df[column].value_counts())
    print("\n" + "-"*40 + "\n")  # Visual separator
'''

# Count how many exact duplicate rows exist before dropping
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_count}")

# Find and print the missing value counts for each column
print("=== Missing Values ===")
print(df.isnull().sum())


                                          categories employmentTypes metadata_expiryDate  metadata_isPostedOnBehalf metadata_jobPostId metadata_newPostingDate metadata_originalPostingDate  metadata_repostCount  metadata_totalNumberJobApplication  metadata_totalNumberOfView  minimumYearsExperience  numberOfVacancies  occupationId    positionLevels                postedCompany_name  salary_maximum  salary_minimum salary_type  status_id status_jobStatus                                              title  average_salary
0  [{"id":13,"category":"Environment / Health"},{...       Permanent          2023-05-08                      False   MCF-2023-0252866              2023-04-08                   2023-03-30                     2                                   5                         151                       0                  1           NaN         Executive               WORKSTONE PTE. LTD.            2800            2000     Monthly          0           Closed  Food Technologist - Cl

In [41]:
# Data cleaning
'''
column analysis and clean plan:
    categories - one job can belong to multiple categories - parse it to replace the original column, also add One-Hot Encoding / Dummy Variables
    metadata_expiryDate, metadata_newPostingDate, metadata_originalPostingDate - convert to datetime format
    metadata_jobPostId - unique identifier for each job post - keep it to avoid duplicates after dropping other columns
    occupationId - no data - can be dropped
    status_id - all 0 - can be dropped
    title - meaningless for analysis - can be dropped
row/cell analysis and clean plan:
    duplicates - check for exact duplicate rows and drop them
    missing values - all missing values are in a single row 197478 (except for one column occupationId which has no data), drop this row 
'''    

# drop duplicates, keep='first' ensures you keep the first occurrence and delete the rest
df = df.drop_duplicates(keep='first')

# Drop columns that are not useful for analysis
columns_to_drop = ['occupationId', 'status_id', 'title']
df = df.drop(columns=columns_to_drop, axis=1)

# Get row indices where any column has a missing value
missing_rows = df[df.isnull().any(axis=1)].index.tolist()
print("Rows with missing values:")
print(missing_rows)

# Drop the row with missing values (row 197478)
df = df.dropna()

#parse categories, add One-Hot Encoding
df['categories'] = df['categories'].apply(ast.literal_eval)
df['categories'] = df['categories'].apply(lambda x: ', '.join([item['category'] for item in x]))
categories_exploded = df['categories'].str.get_dummies(sep=', ')
df = pd.concat([df, categories_exploded], axis=1)

# Convert metadata_expiryDate, metadata_newPostingDate, metadata_originalPostingDate to datetime format
date_cols = [
    "metadata_expiryDate",
    "metadata_newPostingDate",
    "metadata_originalPostingDate",
]
df[date_cols] = df[date_cols].apply(pd.to_datetime)

print("=== Data Cleaning Complete ===")


Rows with missing values:
[197478]
=== Data Cleaning Complete ===


In [42]:
# re-check the db after cleaning

print(df.head())
print(df.shape)   # (rows, columns)

# 1. Structural Description: View column names, non-null counts, and data types
print("--- STRUCTURAL INFO ---")
df.info()

# 2. Statistical Description (Numeric columns): View count, mean, min, max, and percentiles
print("\n--- NUMERIC COLUMNS DESCRIBE ---")
print(df.describe())

# 3. Text/Categorical Description (String columns): View count, unique values, top item, and frequency
print("\n--- TEXT/CATEGORICAL COLUMNS DESCRIBE ---")
print(df.describe(include=['object']))

'''
#unique values and their exact counts for each column
for column in df.columns:
    print(f"=== Unique Values for Column: {column} ===")
    print(df[column].value_counts())
    print("\n" + "-"*40 + "\n")  # Visual separator
'''

# Count how many exact duplicate rows exist before dropping
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicate_count}")

# Find and print the missing value counts for each column
print("=== Missing Values ===")
print(df.isnull().sum())

                                          categories employmentTypes metadata_expiryDate  metadata_isPostedOnBehalf metadata_jobPostId metadata_newPostingDate metadata_originalPostingDate  metadata_repostCount  metadata_totalNumberJobApplication  metadata_totalNumberOfView  minimumYearsExperience  numberOfVacancies    positionLevels                postedCompany_name  salary_maximum  salary_minimum salary_type status_jobStatus  average_salary  Accounting / Auditing / Taxation  Admin / Secretarial  Advertising / Media  Architecture / Interior Design  Banking and Finance  Building and Construction  Consulting  Customer Service  Design  Education and Training  Engineering  Entertainment  Environment / Health  Events / Promotions  F&B  General Management  General Work  Healthcare / Pharmaceutical  Hospitality  Human Resources  Information Technology  Insurance  Legal  Logistics / Supply Chain  Manufacturing  Marketing / Public Relations  Medical / Therapy Services  Others  \
0  Environment 

In [39]:
# save the cleaned DataFrame to a new CSV file
#df.to_csv('SGJobData_cleaned.csv', index=False)
df.to_csv('SGJobData_cleaned.csv.gz', index=False, compression='gzip')